# Feature Engineering
This notebook covers the creation of new features and transformation of existing ones to improve model performance.

## 1. Importing Libraries

In [2]:
import pandas as pd
import numpy as np


## 2. Load Cleaned Data
Loading the cleaned dataset from the previous data cleaning step.

In [3]:
df_clean = pd.read_csv("../data/df_clean.csv")

In [4]:
df_clean.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Credit_History_Missing
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1.0,Urban,Y,0
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N,0
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y,0
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y,0
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y,0


In [5]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Loan_ID                 614 non-null    object 
 1   Gender                  614 non-null    object 
 2   Married                 614 non-null    object 
 3   Dependents              614 non-null    object 
 4   Education               614 non-null    object 
 5   Self_Employed           614 non-null    object 
 6   ApplicantIncome         614 non-null    int64  
 7   CoapplicantIncome       614 non-null    float64
 8   LoanAmount              614 non-null    float64
 9   Loan_Amount_Term        614 non-null    float64
 10  Credit_History          614 non-null    float64
 11  Property_Area           614 non-null    object 
 12  Loan_Status             614 non-null    object 
 13  Credit_History_Missing  614 non-null    int64  
dtypes: float64(4), int64(2), object(8)
memory 

In [6]:
df_clean.isnull().sum()

Loan_ID                   0
Gender                    0
Married                   0
Dependents                0
Education                 0
Self_Employed             0
ApplicantIncome           0
CoapplicantIncome         0
LoanAmount                0
Loan_Amount_Term          0
Credit_History            0
Property_Area             0
Loan_Status               0
Credit_History_Missing    0
dtype: int64

## 3. Drop Loan_ID

In [7]:
df_clean = df_clean.drop("Loan_ID",axis=1)

In [8]:
df_clean.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'Credit_History_Missing'],
      dtype='object')

## 4. Seperate Feature & Target

In [9]:
x = df_clean.drop("Loan_Status",axis=1)   #Feature
y = df_clean["Loan_Status"]               #Target

## 5. Split Train and Test

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

In [12]:
x_train.shape

(491, 12)

In [13]:
x_test.shape

(123, 12)

In [14]:
y_train.shape

(491,)

In [15]:
y_test.shape

(123,)

In [16]:
y_train.value_counts(normalize=True)

Loan_Status
Y    0.686354
N    0.313646
Name: proportion, dtype: float64

In [17]:
y_test.value_counts(normalize=True)

Loan_Status
Y    0.691057
N    0.308943
Name: proportion, dtype: float64

### Verify Stratified Split
Confirming `y_train` and `y_test` maintain the same class proportions as the original dataset after stratified splitting.

## 4. Encoding Categorical Variables
Converting categorical columns into numerical format for model compatibility.

In [18]:
x_train.dtypes

Gender                     object
Married                    object
Dependents                 object
Education                  object
Self_Employed              object
ApplicantIncome             int64
CoapplicantIncome         float64
LoanAmount                float64
Loan_Amount_Term          float64
Credit_History            float64
Property_Area              object
Credit_History_Missing      int64
dtype: object

### Need Encoding:
- Gender
- Married
- Education
- Self_Employed
- Dependents
- Property_Area
### Already Model-Friendly:
- ApplicantIncome
- CoapplicantIncome
- LoanAmount
- Loan_Amount_Term
- Credit_History
- Credit_History_Missing
### Need Target Mapping:
- Loan_Status (Y/N -> 1/0)

### Inspect Unique Values
Checking the unique values of each categorical column before encoding to confirm the exact string values that need to be mapped.

In [19]:
x_train["Gender"].unique()

array(['Male', 'Female'], dtype=object)

In [20]:
x_train["Married"].unique()

array(['No', 'Yes'], dtype=object)

In [21]:
x_train["Education"].unique()

array(['Graduate', 'Not Graduate'], dtype=object)

In [22]:
x_train["Self_Employed"].unique()

array(['No', 'Yes'], dtype=object)

In [23]:
x_train["Dependents"].unique()

array(['0', '1', '2', '3+'], dtype=object)

In [24]:
x_train["Property_Area"].unique()

array(['Urban', 'Semiurban', 'Rural'], dtype=object)

### Label Encoding — Binary Categorical Columns
Mapping binary categorical columns to 0/1 using a dictionary. Applied to both `x_train` and `x_test` using the same mappings to ensure consistency.

In [25]:
mappings = {
    "Gender": {
        "Male": 1,
        "Female": 0
    },
    "Married": {
        "Yes": 1,
        "No": 0
    },
    "Education":{
        "Graduate":1,
        "Not Graduate":0
    },
    "Self_Employed":{
        "Yes":1,
        "No":0
    }
}

In [26]:
for feature in mappings:
    x_train[feature] = x_train[feature].map(mappings[feature])
    x_test[feature] = x_test[feature].map(mappings[feature])

In [27]:
x_train[["Gender","Married","Education","Self_Employed"]].head()

,Gender,Married,Education,Self_Employed
154,1,0,1,0
239,1,1,1,0
448,1,1,1,0
471,1,1,0,0
273,1,1,1,0


In [28]:
x_test[["Gender","Married","Education","Self_Employed"]].head()

,Gender,Married,Education,Self_Employed
150,1,0,1,0
559,0,1,1,0
598,1,1,1,1
235,1,1,1,0
145,0,1,1,0


### Ordinal Encoding — Dependents
`Dependents` has a natural order (0 < 1 < 2 < 3+), so it is encoded ordinally. The `3+` string value is mapped to the integer 3.

In [29]:
dependents_mapping = {
    "0" : 0,
    "1" : 1,
    "2" : 2,
    "3+" : 3
}

In [30]:
x_train["Dependents"] = x_train["Dependents"].map(dependents_mapping)

In [31]:
x_train["Dependents"].unique()

array([0, 1, 2, 3])

In [32]:
x_test["Dependents"] = x_test["Dependents"].map(dependents_mapping)

In [33]:
x_test["Dependents"].unique()

array([0, 1, 3, 2])

### One-Hot Encoding — Property_Area
`Property_Area` has no inherent order (Urban, Semiurban, Rural), so one-hot encoding is used instead of label mapping. `drop_first=True` drops the Rural column to avoid redundancy, since it can be inferred when both remaining columns are 0.

In [34]:
x_train = pd.get_dummies(x_train, columns=["Property_Area"],drop_first=True)

In [35]:
x_test = pd.get_dummies(x_test,columns=["Property_Area"],drop_first=True)

In [36]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 491 entries, 154 to 354
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   491 non-null    int64  
 1   Married                  491 non-null    int64  
 2   Dependents               491 non-null    int64  
 3   Education                491 non-null    int64  
 4   Self_Employed            491 non-null    int64  
 5   ApplicantIncome          491 non-null    int64  
 6   CoapplicantIncome        491 non-null    float64
 7   LoanAmount               491 non-null    float64
 8   Loan_Amount_Term         491 non-null    float64
 9   Credit_History           491 non-null    float64
 10  Credit_History_Missing   491 non-null    int64  
 11  Property_Area_Semiurban  491 non-null    bool   
 12  Property_Area_Urban      491 non-null    bool   
dtypes: bool(2), float64(4), int64(7)
memory usage: 47.0 KB


In [37]:
x_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 123 entries, 150 to 373
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   123 non-null    int64  
 1   Married                  123 non-null    int64  
 2   Dependents               123 non-null    int64  
 3   Education                123 non-null    int64  
 4   Self_Employed            123 non-null    int64  
 5   ApplicantIncome          123 non-null    int64  
 6   CoapplicantIncome        123 non-null    float64
 7   LoanAmount               123 non-null    float64
 8   Loan_Amount_Term         123 non-null    float64
 9   Credit_History           123 non-null    float64
 10  Credit_History_Missing   123 non-null    int64  
 11  Property_Area_Semiurban  123 non-null    bool   
 12  Property_Area_Urban      123 non-null    bool   
dtypes: bool(2), float64(4), int64(7)
memory usage: 11.8 KB


In [38]:
x_train.dtypes

Gender                       int64
Married                      int64
Dependents                   int64
Education                    int64
Self_Employed                int64
ApplicantIncome              int64
CoapplicantIncome          float64
LoanAmount                 float64
Loan_Amount_Term           float64
Credit_History             float64
Credit_History_Missing       int64
Property_Area_Semiurban       bool
Property_Area_Urban           bool
dtype: object

### RobustScaling

### Numerical Scaling — RobustScaler
`RobustScaler` is used instead of `StandardScaler` because `ApplicantIncome` and `LoanAmount` contain significant outliers (as observed in EDA). RobustScaler uses the median and IQR rather than mean and std, making it resistant to the influence of extreme values.

`fit_transform` is applied to `x_train` (learns the scale parameters from training data). `transform` only is applied to `x_test` (uses the same parameters learned from training — no data leakage).

In [39]:
from sklearn.preprocessing import RobustScaler

In [40]:
num_cols=["ApplicantIncome","CoapplicantIncome","LoanAmount","Loan_Amount_Term"]

# Initialize and fit the scaler
scaler = RobustScaler()
x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

### Verify Scaling
Checking the describe output after scaling. The median (50%) should be close to 0 for all columns, confirming the RobustScaler has correctly centered the data around the median.

In [41]:
x_train[num_cols].describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term
count,491.000000,491.000000,491.000000,491.000000
mean,0.572456,0.239865,0.287185,-18.256619
std,2.212328,1.244767,1.323815,65.107943
min,-1.250086,-0.460509,-1.844961,-348.000000
25%,-0.326482,-0.460509,-0.426357,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.673518,0.539491,0.573643,0.000000
max,26.427201,18.132530,8.868217,120.000000


### Encoding Target i.e. Loan_Status

In [42]:
y_train.unique()

array(['Y', 'N'], dtype=object)

In [43]:
y_train.dtypes

dtype('O')

In [44]:
status_mappings = {
    "Y":1,
    "N":0
}

In [45]:
y_train = y_train.map(status_mappings)

In [46]:
y_train.dtypes

dtype('int64')

In [47]:
y_train.head()

154    1
239    1
448    0
471    0
273    1
Name: Loan_Status, dtype: int64

In [48]:
y_test = y_test.map(status_mappings)

In [49]:
y_test.head()

150    0
559    1
598    1
235    1
145    1
Name: Loan_Status, dtype: int64

In [50]:
y_train.unique()

array([1, 0])

In [51]:
y_test.unique()

array([0, 1])

## 6. Save Processed Data
Saving the fully processed train and test sets to the `data/` folder for use in the model building notebook.

In [52]:
x_train.to_csv("../data/x_train.csv", index=False)

In [53]:
x_test.to_csv("../data/x_test.csv", index=False)

In [54]:
y_train.to_csv("../data/y_train.csv", index=False)

In [55]:
y_test.to_csv("../data/y_test.csv", index=False)

## 7. Save Scaler
Saving the fitted `RobustScaler` object so it can be reused during model deployment to scale new incoming data with the exact same parameters learned from training.

In [56]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']